# 0901 Web Search
Gemini + Google Search lookups for the LLM classifier pipeline.
Mirrors `0901_web_search.py` — use this for interactive development and testing.
Results are saved to `09_outputs/web_search_cache.csv` automatically.

In [153]:
import re
import time
import json
from pathlib import Path
from typing import Literal, Optional

import pandas as pd#
from dotenv import load_dotenv
from pydantic import BaseModel, ValidationError
from tqdm.notebook import tqdm

from config import COLUMNS, DEFAULT_INPUT_PATH
from standardization_helpers import (
    normalize_for_key, normalize_employer_for_key, lightly_process_name, lightly_process_employer,
    standardize_city, standardize_state, standardize_occupation_employer,
)

load_dotenv()
from google import genai
from google.genai import types

client = genai.Client()
print("Gemini client ready")

Gemini client ready


## Config

In [154]:
#INPUT_PATH  = DEFAULT_INPUT_PATH
#INPUT_PATH = '../08_alternative_pipeline/08_outputs/pac_classification_input.csv'
INPUT_PATH = '../08_alternative_pipeline/08_outputs/classification_input_combined.csv'
CACHE_PATH  = Path("09_outputs/web_search_cache.csv")
MODEL       = "gemini-2.5-flash"
BATCH_SIZE  = 500
BATCH_INDEX = 0   # increment this between runs: 0, 1, 2, ...
TEST_N      =  None  # set to e.g. 10 for a quick test; overrides BATCH_INDEX/BATCH_SIZE when set

_NOT_EMPLOYED = {
    "not employed", "unknown", "retired", "homemaker", "student", "n/a", "none",
    "unemployed", "housewife", "househusband", "stay at home", "stay-at-home",
    "na", "-", "nan",
}
_BAD_SUMMARIES = {"did not find", ""}

## Helper functions

In [155]:
import pandas as pd, sys
sys.path.insert(0, ".")
from standardization_helpers import NOT_EMPLOYED, lightly_process_employer, standardize_occupation_employer

df = pd.read_csv("../08_alternative_pipeline/08_outputs/pac_classification_input.csv", dtype=str).fillna("")
ind = df[df["entity_type"] == "individual"].copy()

# Mirror pipeline logic for processed employer
ind["_emp_proc"] = [
    lightly_process_employer(std) if std.strip()
    else standardize_occupation_employer(raw.strip().upper())
    for std, raw in zip(ind["standardized_employer_name"], ind["Contributor.Employer"])
]
ind["_occ_proc"] = ind["processed_occupation"].map(lambda o: standardize_occupation_employer(o or ""))

emp_not_employed = ind["_emp_proc"].map(lambda e: not e.strip() or e.strip().lower() in NOT_EMPLOYED)
#occ_employed     = ind["_occ_proc"].map(lambda o: o.strip().lower() not in NOT_EMPLOYED)
occ_employed = ind["_occ_proc"].map(lambda o: bool(o.strip()) and o.strip().lower() not in NOT_EMPLOYED)


mixed = ind[emp_not_employed & occ_employed].copy()
print(f"Employer in NOT_EMPLOYED, occupation not: {len(mixed):,} rows")
print(
    mixed.groupby(["_emp_proc", "_occ_proc"])
         .size()
         .reset_index(name="n")
         .sort_values("n", ascending=False)
         .to_string(index=False)
)


Employer in NOT_EMPLOYED, occupation not: 30 rows
_emp_proc                    _occ_proc  n
  RETIRED                       ARTIST 16
                        PHILANTHROPIST  6
     NONE      INVESTOR/PHILANTHROPIST  5
     NONE INVESTMENTS AND PHILANTHROPY  2
     NONE                     ACTIVIST  1


In [157]:
def _is_not_employed(employer: str, occupation: str) -> bool:
    e = (employer or "").strip().lower()
    o = (occupation or "").strip().lower()
    return (not e or e in _NOT_EMPLOYED) and (not o or o in _NOT_EMPLOYED)


def _make_key(name_raw: str, employer_raw: str, employer_processed: str, occ: str,
              city: str = "", state: str = "", entity_type: str = "individual") -> tuple:
    """
    Composite cache key. normalize_for_key() (uppercase + alphanumeric only) makes
    it stable across processing changes. City+state only for not-employed individuals.
    """
    key_name     = normalize_for_key(name_raw)
    key_employer = normalize_employer_for_key(employer_raw)
    if entity_type == "individual" and _is_not_employed(employer_processed, occ):
        return (key_name, key_employer, normalize_for_key(city), normalize_for_key(state))
    return (key_name, key_employer, "", "")


class IndustrySearchResult(BaseModel):
    industry_summary: str
    urls: list[str]
    confidence: Literal["high", "medium", "low", "unknown"] = "unknown"
    is_prominent: Optional[bool] = None
    prominence_reason: str = ""


search_cache: dict = {}
raw_response_cache: dict = {}
parse_error_names: set = set()

## Load and prepare data

In [158]:
df = pd.read_csv(INPUT_PATH)
str_cols = df.select_dtypes(include=["object", "str"]).columns
df[str_cols] = df[str_cols].fillna("")
print(f"Loaded {len(df):,} rows from {Path(INPUT_PATH).name}")

name_raw_col = COLUMNS["name_raw"]
emp_raw_col  = COLUMNS["employer_raw"]
name_col     = COLUMNS["name"]
employer_col = COLUMNS["employer"]
occ_col      = COLUMNS["occupation"]
city_col     = COLUMNS.get("city")
state_col    = COLUMNS.get("state")
entity_type_col = COLUMNS["entity_type"]

cities = df[city_col].fillna("")  if city_col and city_col in df.columns else pd.Series("", index=df.index)
states = df[state_col].fillna("") if state_col and state_col in df.columns else pd.Series("", index=df.index)

# Lightly-processed names for Gemini (preserves business type / committee language)
df["_search_name"]     = df[name_col].map(lightly_process_name)
# For individuals with a blank standardized employer, fall back to processing the raw
# value so Gemini (and the not-employed check) sees "NONE" rather than blank.
df["_search_employer"] = [
    lightly_process_employer(std) if (std.strip() or etype == "organization")
    else standardize_occupation_employer((raw or "").strip().upper())
    for std, raw, etype in zip(df[employer_col], df[emp_raw_col], df[entity_type_col])
]

# Cache key: normalize_for_key() on raw columns — stable across processing changes
df["_search_key"] = [
    _make_key(nr, er, ep, o, c, s, et)
    for nr, er, ep, o, c, s, et in zip(
        df[name_raw_col], df[emp_raw_col], df["_search_employer"], df[occ_col], cities, states, df[entity_type_col]
    )
]

before = len(df)
df = df.drop_duplicates(subset=["_search_key"]).reset_index(drop=True)
print(f"Dropped {before - len(df):,} duplicate rows — {len(df):,} unique search units")

/var/folders/yd/gc2k9xjj6fz4p6zjd594cdx00000gn/T/ipykernel_10948/4116581289.py:1: DtypeWarning: Columns (0: Recipient.Name, 1: Office, 2: Ballot.Measure, 3: zip_code_processed, 4: race_prop, 5: crosses_threshold, 6: has_contributor_id, 7: name_v1, 8: name_v2, 9: name_v4, 10: needs_review, 11: legacy_match_id, 12: FILER_TYPE, 13: path, 14: is_resolvable, 15: has_id, 16: is_fed_pac, 17: is_pac_no_id, 18: is_self_contribution, 19: unmatched_id, 20: is_pac, 21: is_dues) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_PATH)


Loaded 36,744 rows from classification_input_combined.csv
Dropped 32,781 duplicate rows — 3,963 unique search units


In [ ]:
#df.to_csv("09_outputs/input_sample.csv", index=False)

## Skip already-searched rows (cache check)

In [159]:
df_all = df.copy()
key_strs = df["_search_key"].map(lambda k: "|".join(k))

if CACHE_PATH.exists():
    cache = pd.read_csv(CACHE_PATH)
    cache_str = cache.select_dtypes(include=["object", "str"]).columns
    cache[cache_str] = cache[cache_str].fillna("")
    seen = set(cache["search_key"].dropna())
    n_before = len(df)
    df = df[~key_strs.isin(seen)].reset_index(drop=True)
    key_strs = df["_search_key"].map(lambda k: "|".join(k))
    print(f"Skipped {n_before - len(df):,} rows already in cache — {len(df):,} remaining")
else:
    print(f"No cache found at {CACHE_PATH} — will search all rows")

print(f"\n{len(df):,} rows to search")

Skipped 3,924 rows already in cache — 39 remaining

39 rows to search


In [160]:
## Skip not-employed individuals (no employer or occupation to search on)
ne_mask = df[entity_type_col].eq("individual") & df.apply(
    lambda row: _is_not_employed(row["_search_employer"], row[occ_col]), axis=1
)
n_skipped_ne = ne_mask.sum()
df = df[~ne_mask].reset_index(drop=True)
key_strs = df["_search_key"].map(lambda k: "|".join(k))
print(f"Skipped {n_skipped_ne:,} not-employed individuals — {len(df):,} remaining")


Skipped 39 not-employed individuals — 0 remaining


## Select subset
Set `TEST_N = None` above to run everything, or adjust the sample here.

In [106]:
len(df)

18

In [148]:
if TEST_N is not None:
    sample = df.sample(n=min(TEST_N, len(df)), random_state=42).reset_index(drop=True)
    print(f"TEST run: {len(sample):,} of {len(df):,} rows sampled")
else:
    start  = BATCH_INDEX * BATCH_SIZE
    end    = start + BATCH_SIZE
    sample = df.iloc[start:end].reset_index(drop=True)
    print(f"Batch {BATCH_INDEX}: rows {start}–{min(end, len(df)) - 1} of {len(df)} total ({len(sample)} rows)")

sample = df.copy().reset_index(drop = True)
cities_s = sample[city_col].fillna("")  if city_col and city_col in sample.columns else pd.Series("", index=sample.index)
states_s = sample[state_col].fillna("") if state_col and state_col in sample.columns else pd.Series("", index=sample.index)

rows = list(zip(
    sample["_search_key"], sample["_search_name"], sample["_search_employer"], sample[occ_col],
    cities_s, states_s, sample[entity_type_col],
))
print(f"Batch preview:")
sample[["_search_name", "_search_employer", occ_col]].head(100)

Batch 0: rows 0–-1 of 0 total (0 rows)
Batch preview:


,_search_name,_search_employer,processed_occupation


In [129]:
len(sample)

0

## Search function

In [149]:
def search_industry_gemini(name: str, employer: str, occupation: str = "",
                           city: str = "", state: str = "",
                           not_employed: bool = False) -> tuple[IndustrySearchResult, str]:
    if not_employed:
        user_msg = (
            f"Contributor name: {name}\n"
            f"City: {city or 'Not provided'}\n"
            f"State: {state or 'Not provided'}"
        )
        system_instruction = (
            "You are a research assistant searching for information about California campaign finance donors. "
            "This person lists no employer or occupation. Identifying the specific individual by name alone is difficult — be wary multiple people may share this name. "
            "Use web search to try to find this person. If you can identify them, provide a 1-2 sentence summary of who they are. "
            "If you cannot confidently identify the right person, set industry_summary to 'Unknown'. "
            "Set 'is_prominent' to true if the person appears to be a billionaire, major business executive, "
            "influential political figure, major donor, or someone else notable enough to warrant further review. "
            "Set 'prominence_reason' to a brief explanation if is_prominent is true, otherwise leave it empty. "
            "Return ONLY a raw JSON object with keys: industry_summary, urls, confidence, is_prominent (boolean), prominence_reason (string). "
            "No markdown, no prose -- just the JSON."
        )
    else:
        user_msg = (
            f"Contributor name: {name}\n"
            f"Employer: {employer or 'Non-individual'}\n"
            f"Occupation: {occupation or 'Not provided'}"
        )
        system_instruction = (
            "You are a research assistant identifying California campaign finance contributors. "
            "Use web search to look up the contributor and return a single JSON object with these keys:\n"
            '- "industry_summary": 1-2 sentences describing ONLY the organization\'s or employer\'s business. Do NOT include URLs, confidence ratings, or any metadata in this field.\n'
            '  - For individuals: describe what the employer does.\n'
            '  - For non-individual contributors (employer is "Non-individual"): describe the entity and its purpose.\n'
            '  - If the contributor is a PAC or political committee, note that, and also describe what the PAC supports.\n'
            '  - If you cannot find any information, set to "Unknown".\n'
            '- "urls": list of URLs most useful for this contributor. Empty list if none found.\n'
            '- "confidence": MUST be exactly one of these four strings — no other text: '
            '"high" (clearly identified, detailed info available), '
            '"medium" (clear match but description is short/vague), '
            '"low" (partial match or limited info), "unknown" (nothing found).\n'
            "Return ONLY a raw JSON object with keys: industry_summary, urls, confidence. "
            "No markdown, no prose, no explanation -- just the JSON."
        )

    last_exc = None
    for attempt in range(4):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=user_msg,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    tools=[types.Tool(google_search=types.GoogleSearch())],
                ),
            )
            break
        except Exception as e:
            last_exc = e
            if ("503" in str(e) or "UNAVAILABLE" in str(e)) and attempt < 3:
                wait = 2 ** attempt
                print(f"  503 on attempt {attempt + 1} for {name!r}, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    else:
        raise last_exc

    text = re.sub(r"^```(?:json)?\s*\n?", "", (response.text or "").strip())
    text = re.sub(r"\n?```$", "", text.strip())
    start = text.find("{")
    end   = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        text = text[start: end + 1]

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        parse_error_names.add(name)
        print(f"  Parse error for {name!r}: model returned prose | Raw: {text[:200]}")
        data = None

    if data is not None:
        try:
            if "confidence" in data:
                _conf = str(data["confidence"]).strip().lower()
                _valid = ("high", "medium", "low", "unknown")
                data["confidence"] = next((v for v in _valid if _conf == v or _conf.startswith(v)), "unknown")
            if not not_employed:
                data["is_prominent"] = None
                data["prominence_reason"] = ""
            elif "prominence_reason" not in data:
                data["prominence_reason"] = ""
            return IndustrySearchResult(**data), response.text
        except ValidationError as e:
            parse_error_names.add(name)
            print(f"  Validation error for {name!r}: {e} | Raw: {text[:200]}")

    urls = []
    try:
        for chunk in response.candidates[0].grounding_metadata.grounding_chunks or []:
            if chunk.web and chunk.web.uri:
                urls.append(chunk.web.uri)
    except AttributeError:
        pass

    summary_text = text if data is None else data.get("industry_summary", text)
    return IndustrySearchResult(
        industry_summary=summary_text, urls=urls, confidence="low",
        is_prominent=data.get("is_prominent", None) if (data and not_employed) else None,
        prominence_reason=data.get("prominence_reason", "") if (data and not_employed) else "",
    ), response.text

## Run searches

In [131]:
len(sample)

0

In [109]:
for search_key, name, employer, occ, city, state, entity_type in tqdm(rows, total=len(rows)):
    not_employed = entity_type == "individual" and _is_not_employed(employer, occ)
    try:
        result, raw = search_industry_gemini(
            name, employer, occ, city=city, state=state, not_employed=not_employed,
        )
        search_cache[search_key] = result
        raw_response_cache[search_key] = raw
    except Exception as e:
        print(f"  Error for {name!r} / {employer!r}: {e}")
        search_cache[search_key] = IndustrySearchResult(
            industry_summary="Did not find", urls=[], confidence="unknown"
        )

print(f"\nDone. {len(rows)} searches complete → run 'Save output and update cache'")

  0%|          | 0/18 [00:00<?, ?it/s]


Done. 18 searches complete → run 'Save output and update cache'


## Map results back to sample

In [150]:
def _get(key, field):
    result = search_cache.get(key)
    return getattr(result, field, [] if field == "urls" else "")

sample["industry_summary"]  = sample["_search_key"].map(lambda k: _get(k, "industry_summary"))
sample["urls"]              = sample["_search_key"].map(lambda k: _get(k, "urls"))
sample["confidence"]        = sample["_search_key"].map(lambda k: _get(k, "confidence"))
sample["is_prominent"]      = sample["_search_key"].map(lambda k: _get(k, "is_prominent"))
sample["is_prominent"]      = sample["is_prominent"].map(lambda x: pd.NA if x in (None, "") else x).astype(pd.BooleanDtype())
sample["prominence_reason"] = sample["_search_key"].map(lambda k: _get(k, "prominence_reason"))

# Retry any entries with a bad or empty summary
# ("unknown" is valid for unidentifiable people — don't retry those)
retry_rows = [
    row for row in rows
    if search_cache.get(row[0], IndustrySearchResult(industry_summary="", urls=[])
    ).industry_summary.strip().lower() in _BAD_SUMMARIES
]
print(f"{len(retry_rows)} entries to retry")

0 entries to retry


In [151]:
if retry_rows:
    print(f"Retrying {len(retry_rows)} entries...")
    for search_key, name, employer, occ, city, state, entity_type in tqdm(retry_rows):
        not_employed = entity_type == "individual" and _is_not_employed(employer, occ)
        try:
            result, raw = search_industry_gemini(
                name, employer, occ, city=city, state=state, not_employed=not_employed,
            )
            search_cache[search_key] = result
            raw_response_cache[search_key] = raw
        except Exception as e:
            print(f"  Retry failed for {name!r}: {e}")

    sample["industry_summary"]  = sample["_search_key"].map(lambda k: _get(k, "industry_summary"))
    sample["urls"]              = sample["_search_key"].map(lambda k: _get(k, "urls"))
    sample["confidence"]        = sample["_search_key"].map(lambda k: _get(k, "confidence"))
    sample["is_prominent"]      = sample["_search_key"].map(lambda k: _get(k, "is_prominent"))
    sample["is_prominent"]      = sample["is_prominent"].map(lambda x: pd.NA if x in (None, "") else x).astype(pd.BooleanDtype())
    sample["prominence_reason"] = sample["_search_key"].map(lambda k: _get(k, "prominence_reason"))

In [ ]:
#sample.to_csv("test_web_search.csv", index= False)

## Inspect results

In [88]:
if parse_error_names:
    print(f"Parse errors ({len(parse_error_names)}): {sorted(parse_error_names)[:10]}")

sample[["_search_name", "_search_employer", "industry_summary", "confidence", "is_prominent"]].head(20)

Parse errors (6): ['', 'DIAGEO NORTH AMERICA, INC', 'ERICKSON, DAN', 'HELLMAN PROPERTIES LLC - JERRY TONE', 'JANESCO ENTERPRISES, INC AND AFFILIATED ENTITIES', 'MINC, INC']


,_search_name,_search_employer,industry_summary,confidence,is_prominent
0,AFSCME,,"AFSCME (American Federation of State, County a...",high,<NA>
1,AFSCME,,"The American Federation of State, County and M...",high,<NA>
2,AFSCME,,"AFSCME (American Federation of State, County a...",high,<NA>
3,ECA SOLAR,,"ECA SOLAR is a developer, engineer, installer,...",high,<NA>
4,RENEWABLE PROPERTIES,,RENEWABLE PROPERTIES is a developer and invest...,high,<NA>
5,NEW ENERGY EQUITY,,New Energy Equity is a company that specialize...,high,<NA>
6,COMMON ENERGY,,Common Energy is a renewable energy company th...,high,<NA>
7,"THE WONDERFUL COMPANY, LLC",,"The Wonderful Company, LLC is a privately held...",high,<NA>
8,"CRC SERVICES, LLC",,"CRC Services, LLC is a vocational consulting a...",high,<NA>
9,"VALERO SERVICES, INC AND AFILIATED ENTITIES",,"Valero Services, Inc. is an affiliate of Valer...",high,<NA>


In [89]:
# Inspect a specific contributor
name_to_check = "ERICKSON, DAN"  # fill in e.g. "SMITH, JOHN"
if name_to_check:
    hit = sample[sample["_search_name"] == name_to_check]
    if not hit.empty:
        key = hit.iloc[0]["_search_key"]
        result = search_cache.get(key)
        raw = raw_response_cache.get(key, "NOT IN RAW CACHE")
        print(f"Name: {hit.iloc[0]['_search_name']} | Employer: {hit.iloc[0]['_search_employer']}")
        print(f"Summary: {result.industry_summary if result else 'N/A'}")
        print(f"Confidence: {result.confidence if result else 'N/A'}")
        print(f"Prominent: {result.is_prominent} — {result.prominence_reason if result else ''}")
        print(f"Raw: {raw[:500]}")
    else:
        print(f"{name_to_check!r} not found in sample")

Name: ERICKSON, DAN | Employer: DBA: DAN ERICKSON
Summary: 
Confidence: low
Prominent: None — 


TypeError: 'NoneType' object is not subscriptable

# Use Cache Only

In [161]:
# Use when df is empty after cache filtering (all entities already cached).
# Produces the same output format as a normal search batch, without any API calls.
from datetime import date

assert len(df) == 0, "df is not empty — run normal search flow instead"

# Load cache into a lookup dict (mirrors search_cache but from disk)
cache_df = pd.read_csv(CACHE_PATH, dtype=str).fillna("")
cache_lookup = {
    row["search_key"]: row
    for _, row in cache_df.iterrows()
}

def _get_cached(key_tuple, field):
    key_str = "|".join(key_tuple)
    return cache_lookup.get(key_str, {}).get(field, "")

sample = df_all.copy()
sample["industry_summary"]  = sample["_search_key"].map(lambda k: _get_cached(k, "industry_summary"))
sample["urls"]              = sample["_search_key"].map(lambda k: _get_cached(k, "urls"))
sample["confidence"]        = sample["_search_key"].map(lambda k: _get_cached(k, "confidence"))
sample["is_prominent"]      = sample["_search_key"].map(lambda k: _get_cached(k, "is_prominent"))
sample["prominence_reason"] = sample["_search_key"].map(lambda k: _get_cached(k, "prominence_reason"))
sample["search_key"]        = sample["_search_key"].map(lambda k: "|".join(k))

cache_hit_rate = sample["industry_summary"].replace("", pd.NA).notna().mean()
print(f"Cache hits: {cache_hit_rate:.1%} of {len(sample):,} entities")

# Skip to "Save output" section from here — don't run the search or retry cells


Cache hits: 98.8% of 3,963 entities


In [163]:
print(len(df_all))
print(len(sample))

3963
3963


## Save output and update cache

In [164]:
from datetime import date

out_dir = Path("09_outputs")
out_dir.mkdir(exist_ok=True)

#DON'T FORGET TO SET
output_path = out_dir / f"web_search_contributors_combined_all_{date.today().isoformat()}.csv"
#output_path = out_dir / f"web_search_major_donors_missed_{BATCH_INDEX}_{date.today().isoformat()}.csv"
#output_path = out_dir / f"web_search_full_no_batch_{date.today().isoformat()}.csv"
#output_path = out_dir / f"web_search_pac_contrib_batch_{BATCH_INDEX}_{date.today().isoformat()}.csv"
#output_path = out_dir / f"web_search_25_test_{date.today().isoformat()}.csv"

sample["search_key"] = sample["_search_key"].map(lambda k: "|".join(k))
sample_out = sample.drop(columns=["_search_key", "_search_name", "_search_employer"])
sample_out.to_csv(output_path, index=False)
print(f"Saved batch {BATCH_INDEX}: {len(sample_out):,} rows → {output_path}")

Saved batch 0: 3,963 rows → 09_outputs/web_search_contributors_combined_all_2026-08-16.csv


## Update cache
Run this only after inspecting all batches and you're satisfied with the results.

In [120]:
# UPDATE CACHE FROM SEARCH JUST RAN
_CACHE_COLS = [
    "search_key",
    COLUMNS["name_raw"], COLUMNS["employer_raw"],
    COLUMNS["name"], COLUMNS["employer"],
    COLUMNS["city"], COLUMNS["state"],
    "industry_summary", "urls", "confidence", "is_prominent", "prominence_reason",
]
new_cache_rows = sample_out[[c for c in _CACHE_COLS if c in sample_out.columns]].copy()
if CACHE_PATH.exists():
    existing = pd.read_csv(CACHE_PATH)
    updated = pd.concat([existing, new_cache_rows], ignore_index=True)
    updated = updated.drop_duplicates(subset=["search_key"], keep="last")
else:
    CACHE_PATH.parent.mkdir(exist_ok=True)
    updated = new_cache_rows
updated.to_csv(CACHE_PATH, index=False)
print(f"Cache updated: {len(updated):,} total entries")

Cache updated: 4,100 total entries


In [ ]:
# UPDATE CACHE FROM CSV
_CACHE_COLS = [
    "search_key",
    COLUMNS["name_raw"], COLUMNS["employer_raw"],
    COLUMNS["name"], COLUMNS["employer"],
    COLUMNS["city"], COLUMNS["state"],
    "industry_summary", "urls", "confidence", "is_prominent", "prominence_reason",
]

#batch_file = Path("09_outputs/web_search_batch_0_2026-08-12.csv")
batch_file = Path("09_outputs/web_search_pac_contrib_batch_0_2026-08-13.csv")
batch_df = pd.read_csv(batch_file)
new_cache_rows = batch_df[[c for c in _CACHE_COLS if c in batch_df.columns]].copy()

existing = pd.read_csv(CACHE_PATH) if CACHE_PATH.exists() else pd.DataFrame(columns=_CACHE_COLS)
n_before = len(existing)
updated = pd.concat([existing, new_cache_rows], ignore_index=True)
updated = updated.drop_duplicates(subset=["search_key"], keep="last")
updated.to_csv(CACHE_PATH, index=False)
print(f"Cache: {n_before:,} -> {len(updated):,} entries ({len(updated) - n_before:,} new)")


Cache: 1,696 -> 2,696 entries (1,000 new)


## Scratch

In [ ]:
import pandas as pd, sys
sys.path.insert(0, ".")
from standardization_helpers import (
    normalize_for_key, normalize_employer_for_key, NOT_EMPLOYED,
    lightly_process_employer, standardize_occupation_employer, standardize_city,
)

def _is_not_employed(employer, occ):
    e = (employer or "").strip().lower()
    o = (occ or "").strip().lower()
    return (not e or e in NOT_EMPLOYED) and (not o or o in NOT_EMPLOYED)


def make_key(row):
    name      = normalize_for_key(row.get("Contributor.Name", "") or "")
    employer  = normalize_employer_for_key(row.get("Contributor.Employer", "") or "")
    etype     = (row.get("entity_type", "") or "").strip()
    std_emp   = (row.get("standardized_employer_name", "") or "").strip()
    raw_emp   = (row.get("Contributor.Employer", "") or "").strip()
    emp_proc  = (lightly_process_employer(std_emp) if (std_emp or etype == "organization")
                 else standardize_occupation_employer(raw_emp.upper()))
    occ       = (row.get("processed_occupation", "") or "")
    std_city  = (row.get("standardized_city", "") or "").strip()
    raw_city  = (row.get("Contributor.City", "") or "").strip()
    city      = normalize_for_key(standardize_city(std_city) if std_city else standardize_city(raw_city))
    state     = normalize_for_key(row.get("Contributor.State", "") or "")
    if etype == "individual" and _is_not_employed(emp_proc, occ):
        return f"{name}|{employer}|{city}|{state}"
    return f"{name}|{employer}||"

df      = pd.read_csv("../08_alternative_pipeline/08_outputs/classification_input.csv", dtype=str).fillna("")
already = pd.read_csv("../08_alternative_pipeline/08_inputs/already_classified_contributions.csv", dtype=str).fillna("")
cache   = pd.read_csv("09_outputs/web_search_cache.csv", dtype=str).fillna("")

in_scope = df[df["uuid"].isin(already["uuid"])].copy()
in_scope["_key"] = in_scope.apply(make_key, axis=1)
cache_keys = set(cache["search_key"].str.strip())

in_scope["in_cache"] = in_scope["_key"].isin(cache_keys)
hit, miss = in_scope["in_cache"].sum(), (~in_scope["in_cache"]).sum()
print(f"Matched: {len(in_scope):,}  |  hits: {hit:,}  |  misses: {miss:,}")
in_scope[~in_scope["in_cache"]][['Contributor.Name','Contributor.Employer','_key']]
